# Model characteristics
- Module validity verification
- Robustness investigation

In [ ]:
import importlib
import json
import sys
from pathlib import Path

import numpy as np
import torch
from tqdm.notebook import tqdm


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "generator").is_dir() and (path / "methods").is_dir():
            return path
    raise RuntimeError("Cannot find project root. Please run this notebook inside the project directory.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.flow import FlowManager
from generator.images import config_events
from generator.loader import EventManager
from methods.affinegs.cmax import ContrastMaximizationLoss
from methods.affinegs.gaussian_flow.model import GaussianVelocityField_Tiled
from methods.affinegs.gaussian_flow.utils import divergence_regularizer
import methods.affinegs.gaussian_flow.config as gf_config
import methods.affinegs.gaussian_flow.training as gf_training
from utils.add_noise import add_event_noise
from utils.metrics import PIVmetric
from utils.painter import extract_data, plot_curvegraph, plot_module_validity

GaussianFlowConfig = gf_config.GaussianFlowConfig
run_gaussian_flow = gf_training.run_gaussian_flow
train_gaussian_flow = gf_training.train_gaussian_flow
render_gaussian_flow = gf_training.render_gaussian_flow

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "output" / "EXP1"
DEFAULT_PLOT_SIZE = [10, 10, 12]
DEFAULT_USE_CHINESE = False
JACOBIAN_X_MAJOR_TICKS = list(range(100, 501, 100))
JACOBIAN_X_MINOR_TICKS = list(range(50, 501, 50))


def save_results_to_json(results, save_path, indent=1):
    def convert_numpy_types(obj):
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    try:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=indent, default=convert_numpy_types)
        print(f"Results saved to: {save_path}")
    except Exception as e:
        print(f"Save failed: {e}")


def load_results_json(filename):
    with open(OUTPUT_DIR / filename, "r", encoding="utf-8") as f:
        return json.load(f)


def trimmed_mean(values, trim=1, digits=4):
    values = np.sort(np.asarray(values, dtype=float))
    if trim > 0 and values.size > 2 * trim:
        values = values[trim:-trim]
    return round(float(np.mean(values)), digits)


def set_all_seeds(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)


def reload_gaussian_flow_modules():
    global GaussianFlowConfig, run_gaussian_flow, train_gaussian_flow, render_gaussian_flow
    importlib.reload(gf_config)
    importlib.reload(gf_training)
    GaussianFlowConfig = gf_config.GaussianFlowConfig
    run_gaussian_flow = gf_training.run_gaussian_flow
    train_gaussian_flow = gf_training.train_gaussian_flow
    render_gaussian_flow = gf_training.render_gaussian_flow


def make_simulated_event_data(flow_index, *, time_length, d, density, t_min=0, t_max=3):
    cfg_evts = config_events()
    cfg_evts.time_length = time_length
    cfg_evts.d = d
    cfg_evts.density = density

    event_loader = EventManager(
        source_type="simulator",
        path=None,
        cfg=cfg_evts,
        flow_func=flow_funcs[flow_index].evaluate,
    )
    evts, (u, v) = event_loader.load()
    events = evts[(evts[:, 2] > t_min) & (evts[:, 2] < t_max)]
    return events, u.cpu().numpy(), v.cpu().numpy()


def metric_axis_config(y_data_list, tick_step=0.2):
    vals = np.concatenate([np.asarray(y, dtype=float) for y in y_data_list])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None, None

    y_max = float(vals.max())
    upper = np.ceil((y_max + 1e-9) / tick_step) * tick_step
    upper = max(tick_step, upper)
    if upper - y_max < 0.05 * tick_step:
        upper += tick_step
    upper = round(float(upper), 2)
    ticks = list(np.round(np.arange(0, upper + 0.5 * tick_step, tick_step), 2))
    return (0, upper), ticks


def plot_metric_pair(
    x_data,
    y_rmse_list,
    y_aee_list,
    *,
    prefix,
    xlabel_en,
    xlabel_zh,
    xlim,
    x_major_ticks,
    x_minor_ticks,
    rmse_ylim=None,
    aee_ylim=None,
    rmse_y_major_ticks=None,
    aee_y_major_ticks=None,
    save=True,
):
    paths = {
        "rmse": str(OUTPUT_DIR / f"{prefix}-rmse.pdf") if save else None,
        "aee": str(OUTPUT_DIR / f"{prefix}-aee.pdf") if save else None,
    }

    for metric, y_data_list, ylim, y_major_ticks in [
        ("RMSE", y_rmse_list, rmse_ylim, rmse_y_major_ticks),
        ("AEE", y_aee_list, aee_ylim, aee_y_major_ticks),
    ]:
        plot_curvegraph(
            x_data=x_data,
            y_data_list=y_data_list,
            legend_names_en=flow_names,
            legend_names_zh=None,
            title_en=None,
            title_zh=None,
            xlabel_en=xlabel_en,
            xlabel_zh=xlabel_zh,
            ylabel_en=f"{metric} $[px/ms]$",
            ylabel_zh=f"{metric} $[px/ms]$",
            ticks_size=DEFAULT_PLOT_SIZE[0],
            legend_size=DEFAULT_PLOT_SIZE[1],
            label_size=DEFAULT_PLOT_SIZE[2],
            xlim=xlim,
            ylim=ylim,
            x_major_ticks=x_major_ticks,
            x_minor_ticks=x_minor_ticks,
            y_major_ticks=y_major_ticks,
            y_minor_ticks=None,
            x_log=False,
            filename=paths[metric.lower()],
            use_chinese=DEFAULT_USE_CHINESE,
        )


def plot_jacobian_ablation(data_filename, output_prefix, x_label, save=True):
    data = load_results_json(data_filename)
    for metric in ["rmse", "aee"]:
        save_path = str(OUTPUT_DIR / f"{output_prefix}-{metric}.pdf") if save else None
        plot_module_validity(
            data,
            flow_names=flow_names,
            metric=metric,
            save_path=save_path,
            x_label=x_label,
            style_names={"1": "Affine", "0": "Const"},
            style_linestyles={"1": "-", "0": "--"},
            style_legend_anchor=(0.58, 1.02),
            figsize=(5, 3),
            xlim=(40, 510),
            x_major_ticks=JACOBIAN_X_MAJOR_TICKS,
            x_minor_ticks=JACOBIAN_X_MINOR_TICKS,
            ticks_size=10,
            legend_size=10,
            label_size=12,
            legend_mode="combined",
            combined_legend_loc="upper right",
            combined_legend_anchor=(1.0, 1.02),
            combined_legend_ncol=1,
            combined_legend_label_mode="method_case",
        )


def extract_div_ratio_sweep(data, metric):
    x_vals = None
    y_list = []
    for flow_name in flow_names:
        ratios = np.array(sorted(float(k) for k in data[flow_name].keys()), dtype=float)
        values = np.array([data[flow_name][f"{r:.2f}"][metric] for r in ratios], dtype=float)
        if x_vals is None:
            x_vals = ratios
        y_list.append(values)
    return x_vals, y_list


In [ ]:
FLOW_DATA_PATHS = [
    PROJECT_ROOT / "data" / "datasets" / "JHTDB_isotropic1024_hd_00001_flow.npz",
    PROJECT_ROOT / "data" / "datasets" / "DNS_turbulence_00001_flow.npz",
    PROJECT_ROOT / "data" / "datasets" / "SQG_00001_flow.npz",
]
flow_names = ["JHTDB-isotropic", "DNS-turbulence", "SQG"]
flow_funcs = [FlowManager(mode="data", data_path=str(path)) for path in FLOW_DATA_PATHS]
flow0, flow1, flow2 = flow_funcs


## Module validity verification

- Same Gaussian quantity

In [ ]:
jacobian_results = {}
jacobian_init_ns = list(range(50, 510, 50))
jacobian_repeats = 10
jacobian_configs = [
    (enable_jacobian, init_n)
    for enable_jacobian in [1, 0]
    for init_n in jacobian_init_ns
]

for flow_index in tqdm(range(3), desc="Flows", position=0):
    flow_key = f"flow{flow_index}"
    jacobian_results[flow_key] = {}
    events, h_gt, w_gt = make_simulated_event_data(flow_index, time_length=5, d=0.5, density=0.05)

    for enable_jacobian, init_n in tqdm(jacobian_configs, desc=flow_key, leave=False, position=1):
        jac_key = str(enable_jacobian)
        jacobian_results[flow_key].setdefault(jac_key, {})
        rmse_list, aee_list = [], []

        for _ in tqdm(range(jacobian_repeats), desc="repeat", leave=False, position=2):
            cfg = GaussianFlowConfig(
                verbose=False,
                init_n=init_n,
                enable_jacobian=enable_jacobian,
                phase=[500, 200, 250],
                iterations=300,
                lr_j=0.001,
            )
            res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
            pred_h, pred_w = res.flow[0], res.flow[1]
            rmse, aee = PIVmetric(pred_h, pred_w, h_gt, w_gt, show=False)[:2]
            rmse_list.append(rmse)
            aee_list.append(aee)

        jacobian_results[flow_key][jac_key][str(init_n)] = {
            "rmse": trimmed_mean(rmse_list),
            "aee": trimmed_mean(aee_list),
        }

save_results_to_json(jacobian_results, OUTPUT_DIR / "1-jacobian-results.json", indent=1)


In [ ]:
save_jacobian_figures = True

plot_jacobian_ablation(
    data_filename="1-jacobian-results.json",
    output_prefix="1-jacobian",
    x_label="Number of Gaussians (init_n)",
    save=save_jacobian_figures,
)


- Same Gaussian quantity with frozen geometry


In [ ]:
def clone_shared_geometry_model(base_model, enable_jacobian, cfg):
    """Create a model with exactly the same Gaussian geometry as base_model."""
    model = GaussianVelocityField_Tiled(
        num_gaussians=base_model.num_gaussians,
        H=base_model.H,
        W=base_model.W,
        tile_size=base_model.tile_size,
        enable_jacobian=enable_jacobian,
        device=base_model.mu.device,
        jacobian_mode=cfg.jacobian_mode,
        init_scale_h=cfg.init_scale_h,
        init_scale_w=cfg.init_scale_w,
    ).to(base_model.mu.device)

    with torch.no_grad():
        model.mu.copy_(base_model.mu)
        model.log_scale.copy_(base_model.log_scale)
        model.rotation.copy_(base_model.rotation)
        model.opacity_logit.copy_(base_model.opacity_logit)
        model.local_velocity.copy_(base_model.local_velocity)
        if model.J_params is not None:
            model.J_params.zero_()

    # Freeze only the Gaussian support. The ablation tests the velocity model on identical geometry.
    for param in (model.mu, model.log_scale, model.rotation, model.opacity_logit):
        param.requires_grad_(False)

    model.rebuild_indexer()
    return model


def train_frozen_geometry_model(model, events, cfg, iterations):
    model.train()
    events = events.to(model.mu.device)

    param_groups = [{"params": [model.local_velocity], "lr": 0.05, "name": "velocity"}]
    if model.J_params is not None:
        lr_j = cfg.lr_j if cfg.lr_j is not None else 0.001
        param_groups.append({"params": [model.J_params], "lr": lr_j, "name": "jacobian"})

    optimizer = torch.optim.Adam(param_groups, betas=(0.9, 0.999), eps=1e-15)
    cmax_loss_fn = ContrastMaximizationLoss(
        loss_type=cfg.cmax_loss_type,
        sigma=cfg.cmax_sigma,
        kernel_size=cfg.cmax_kernel_size,
    ).to(model.mu.device)

    for i in range(iterations):
        optimizer.zero_grad(set_to_none=True)
        pred_flow, _, _, _ = model(rebuild_index=(i == 0), compute_derivatives=False)
        loss_data = cmax_loss_fn(events, pred_flow, (cfg.H, cfg.W))
        div_loss = divergence_regularizer(pred_flow, top_ratio=cfg.div_top_ratio)
        weight_div = cfg.w_div if i < cfg.div_last else 0.0
        loss = loss_data + weight_div * div_loss
        loss.backward()
        optimizer.step()

    return model


frozen_geometry_results = {}
frozen_init_ns = list(range(50, 510, 50))
frozen_repeats = 10
frozen_pretrain_iterations = 300
frozen_refine_iterations = 150

for flow_index in tqdm(range(3), desc="Flows", position=0):
    flow_key = f"flow{flow_index}"
    frozen_geometry_results[flow_key] = {"1": {}, "0": {}}
    events, h_gt, w_gt = make_simulated_event_data(flow_index, time_length=5, d=0.5, density=0.05)

    for init_n in tqdm(frozen_init_ns, desc=flow_key, leave=False, position=1):
        metric_cache = {
            "1": {"rmse": [], "aee": []},
            "0": {"rmse": [], "aee": []},
        }

        for _ in tqdm(range(frozen_repeats), desc="repeat", leave=False, position=2):
            base_cfg = GaussianFlowConfig(
                verbose=False,
                init_n=init_n,
                enable_jacobian=False,
                phase=[10**9, 10**9, 10**9],
                iterations=frozen_pretrain_iterations,
                lr_j=0.001,
            )
            base_model, _ = train_gaussian_flow(events=events, cfg=base_cfg, flowgt=None)

            for jac_key, enable_jacobian in [("1", True), ("0", False)]:
                refine_cfg = GaussianFlowConfig(
                    verbose=False,
                    init_n=init_n,
                    enable_jacobian=enable_jacobian,
                    phase=[10**9, 10**9, 10**9],
                    iterations=frozen_refine_iterations,
                    lr_j=0.001,
                )
                model = clone_shared_geometry_model(base_model, enable_jacobian, refine_cfg)
                model = train_frozen_geometry_model(model, events, refine_cfg, frozen_refine_iterations)
                res = render_gaussian_flow(model, verbose=False)

                pred_h, pred_w = res.flow[0], res.flow[1]
                rmse, aee = PIVmetric(pred_h, pred_w, h_gt, w_gt, show=False)[:2]
                metric_cache[jac_key]["rmse"].append(rmse)
                metric_cache[jac_key]["aee"].append(aee)

                del model, res
                torch.cuda.empty_cache()

            del base_model
            torch.cuda.empty_cache()

        for jac_key in ["1", "0"]:
            frozen_geometry_results[flow_key][jac_key][str(init_n)] = {
                "rmse": trimmed_mean(metric_cache[jac_key]["rmse"]),
                "aee": trimmed_mean(metric_cache[jac_key]["aee"]),
                "init_n": int(init_n),
                "frozen_geometry": True,
                "pretrain_iterations": int(frozen_pretrain_iterations),
                "refine_iterations": int(frozen_refine_iterations),
                "repeats": int(frozen_repeats),
            }

save_results_to_json(frozen_geometry_results, OUTPUT_DIR / "2-jacobian-frozen-geometry-results.json", indent=1)


In [ ]:
save_frozen_geometry_figures = True

plot_jacobian_ablation(
    data_filename="2-jacobian-frozen-geometry-results.json",
    output_prefix="2-jacobian-frozen-geometry",
    x_label="Number of Gaussians (shared non-velocity params)",
    save=save_frozen_geometry_figures,
)


## Robustness investigation

- particle density

In [ ]:
density_results = {}
density_values = np.round(np.arange(0.01, 0.1001, 0.01), 2)
density_repeats = 10

for flow_index in tqdm(range(3), desc="flows"):
    single_flow_results = {}

    for density in tqdm(density_values, desc=f"{flow_names[flow_index]}-Density", leave=False):
        rmse_list, aee_list = [], []

        for _ in range(density_repeats):
            events, u_gt, v_gt = make_simulated_event_data(
                flow_index,
                time_length=4,
                d=0.05,
                density=float(density),
            )
            cfg = GaussianFlowConfig(lr_j=0.001)
            result = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)

            pred_h, pred_w = result.flow[0], result.flow[1]
            rmse, aee = PIVmetric(u_gt, v_gt, pred_h, pred_w, show=False)[:2]
            rmse_list.append(rmse)
            aee_list.append(aee)

        single_flow_results[f"{density:.6f}"] = {
            "rmse": trimmed_mean(rmse_list),
            "aee": trimmed_mean(aee_list),
        }

    density_results[flow_names[flow_index]] = single_flow_results

save_results_to_json(density_results, OUTPUT_DIR / "3-density-results.json", indent=1)


In [ ]:
save_density_figures = True

density_data = load_results_json("3-density-results.json")
x_vals, y_rmse_list, y_aee_list = extract_data(density_data)

rmse_ylim, rmse_y_major_ticks = metric_axis_config(y_rmse_list, tick_step=0.2)
aee_ylim, aee_y_major_ticks = metric_axis_config(y_aee_list, tick_step=0.2)
density_sample_ticks = list(np.round(np.arange(0.01, 0.1001, 0.01), 2))

plot_metric_pair(
    x_data=x_vals,
    y_rmse_list=y_rmse_list,
    y_aee_list=y_aee_list,
    prefix="3-density",
    xlabel_en="Particle density $\rho_{p}$ $[ppp]$",
    xlabel_zh="粒子密度 $\rho_{p}$",
    xlim=(0.008, 0.102),
    x_major_ticks=list(np.round(np.arange(0.01, 0.1001, 0.02), 2)),
    x_minor_ticks=density_sample_ticks,
    rmse_ylim=rmse_ylim,
    aee_ylim=aee_ylim,
    rmse_y_major_ticks=rmse_y_major_ticks,
    aee_y_major_ticks=aee_y_major_ticks,
    save=save_density_figures,
)


- noise ratio

In [ ]:
noise_results = {}
noise_ratios = np.arange(0, 0.21, 0.02)
noise_repeats = 10

for flow_index in tqdm(range(3), desc="flows"):
    events, u_gt, v_gt = make_simulated_event_data(flow_index, time_length=5, d=0.5, density=0.05)
    single_flow_results = {}

    for noise_ratio in tqdm(noise_ratios, desc="Noise ratio", leave=False):
        rmse_list, aee_list = [], []

        for _ in range(noise_repeats):
            noisy_events = add_event_noise(events, 256, 256, noise_ratio=float(noise_ratio))
            cfg = GaussianFlowConfig()
            res = run_gaussian_flow(events=noisy_events, cfg=cfg, flowgt=None)

            pred_h, pred_w = res.flow[:2]
            rmse, aee = PIVmetric(u_gt, v_gt, pred_h, pred_w, show=False)[:2]
            rmse_list.append(float(rmse))
            aee_list.append(float(aee))

        single_flow_results[f"{noise_ratio:.2f}"] = {
            "rmse": trimmed_mean(rmse_list),
            "aee": trimmed_mean(aee_list),
        }

    noise_results[flow_names[flow_index]] = single_flow_results

save_results_to_json(noise_results, OUTPUT_DIR / "4-noise-results.json", indent=1)


In [ ]:
save_noise_figures = True

noise_data = load_results_json("4-noise-results.json")
x_vals, y_rmse_list, y_aee_list = extract_data(noise_data)

plot_metric_pair(
    x_data=x_vals * 100,
    y_rmse_list=y_rmse_list,
    y_aee_list=y_aee_list,
    prefix="4-noise",
    xlabel_en="Noise ratio $R$ $[\%]$",
    xlabel_zh="噪声比例 $R$ $[\%]$",
    xlim=(-0.5, 20.5),
    x_major_ticks=list(range(0, 21, 5)),
    x_minor_ticks=list(range(1, 21, 1)),
    rmse_ylim=(0, 0.6),
    aee_ylim=(0, 0.6),
    rmse_y_major_ticks=[0, 0.2, 0.4, 0.6],
    aee_y_major_ticks=[0, 0.2, 0.4, 0.6],
    save=save_noise_figures,
)


## Divergence top-ratio sweep

This experiment varies the percentage of the largest-divergence responses used by the weak divergence regularizer from 0% to 20% in 1% increments.


In [ ]:
reload_gaussian_flow_modules()

div_ratio_sweep_results = {}
div_top_ratios = np.round(np.arange(0.0, 0.2001, 0.02), 2)
num_repeats_ratio = 5
ratio_base_seed = 20260710

for flow_index in tqdm(range(3), desc="Flows"):
    flow_name = flow_names[flow_index]
    div_ratio_sweep_results[flow_name] = {}

    set_all_seeds(ratio_base_seed + flow_index)
    events, u_gt, v_gt = make_simulated_event_data(flow_index, time_length=5, d=0.5, density=0.05)

    for ratio in tqdm(div_top_ratios, desc=f"{flow_name} ratios", leave=False):
        rmse_list, aee_list = [], []

        for repeat_index in range(num_repeats_ratio):
            # Paired seed across ratios: only div_top_ratio changes.
            run_seed = ratio_base_seed + flow_index * 100 + repeat_index
            set_all_seeds(run_seed)

            cfg = GaussianFlowConfig(
                div_top_ratio=float(ratio),
                lr_j=0.001,
                verbose=False,
            )
            result = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
            pred_h, pred_w = result.flow[:2]
            rmse, aee = PIVmetric(u_gt, v_gt, pred_h, pred_w, show=False)[:2]
            rmse_list.append(float(rmse))
            aee_list.append(float(aee))

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        div_ratio_sweep_results[flow_name][f"{ratio:.2f}"] = {
            "rmse": round(float(np.mean(rmse_list)), 4),
            "aee": round(float(np.mean(aee_list)), 4),
            "std_rmse": round(float(np.std(rmse_list)), 4),
            "std_aee": round(float(np.std(aee_list)), 4),
            "num_repeats": int(num_repeats_ratio),
        }

save_results_to_json(div_ratio_sweep_results, OUTPUT_DIR / "5-div-top-ratio-sweep-results.json", indent=1)


In [ ]:
save_div_ratio_figures = True

div_ratio_sweep_data = load_results_json("5-div-top-ratio-sweep-results.json")
x_vals, y_rmse_list = extract_div_ratio_sweep(div_ratio_sweep_data, "rmse")
_, y_aee_list = extract_div_ratio_sweep(div_ratio_sweep_data, "aee")

plot_metric_pair(
    x_data=x_vals * 100,
    y_rmse_list=y_rmse_list,
    y_aee_list=y_aee_list,
    prefix="5-div-top-ratio-sweep",
    xlabel_en="Top-divergence ratio $[\%]$",
    xlabel_zh="最大散度比例 $[\%]$",
    xlim=(-0.5, 20.5),
    x_major_ticks=list(range(0, 21, 2)),
    x_minor_ticks=list(range(1, 21, 2)),
    save=save_div_ratio_figures,
)
